In [9]:
import cv2, os, math, random
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters
from scipy.interpolate import splprep, splev

In [10]:
LOAD_FOLDER="png"
SAVE_FOLDER="sketch" #from here

In [11]:
def augment_contour_points(cnt, max_seg_length=10):
    """
    Given a contour (cnt), if a segment's length is greater than or equal to max_seg_length, insert an additional point in the middle of the segment.
    """
    new_cnt = []
    cnt = cnt.squeeze()
    n = len(cnt)

    for i in range(n):
        p1 = cnt[i]
        p2 = cnt[(i + 1) % n]
        new_cnt.append(p1)

        dist = np.linalg.norm(p2 - p1)
        if dist > max_seg_length:
            n_inserts = int(dist // max_seg_length)
            for j in range(1, n_inserts + 1):
                interp = p1 + (p2 - p1) * (j / (n_inserts + 1))
                new_cnt.append(interp.astype(np.int32))

    return np.array(new_cnt, dtype=np.int32).reshape(-1, 1, 2)

In [12]:

SLASH_COLOR = (0, 0, 0)  # BGR
BAT_COLOR = (0, 0, 0)  # BGR


In [13]:
def clearsketch(img, thresh=93, thickness=2, blurrange=15, e=0.001, batshape="random", fillshape="random"):
    #preprocessing
    img_small = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LANCZOS4)
    img_gray = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY)
    _, mask_bat = cv2.threshold(img_gray, 60, 255, cv2.THRESH_BINARY)

    grid_size = 16
    cell_height = mask_bat.shape[0] // grid_size
    cell_width = mask_bat.shape[1] // grid_size

    for i in range(grid_size):
        y = i * cell_height
        cv2.line(mask_bat, (0, y), (mask_bat.shape[1], y), color=255, thickness=1)
        x = i * cell_width
        cv2.line(mask_bat, (x, 0), (x, mask_bat.shape[0]), color=255, thickness=1)
    cv2.line(mask_bat, (0, mask_bat.shape[0] - 1), (mask_bat.shape[1], mask_bat.shape[0] - 1), color=255, thickness=1)
    cv2.line(mask_bat, (mask_bat.shape[1] - 1, 0), (mask_bat.shape[1] - 1, mask_bat.shape[0]), color=255, thickness=1)

    batContours, _ = cv2.findContours(mask_bat, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    _, mask_wall = cv2.threshold(img_gray, thresh, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask_wall, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    canvas_1 = np.ones((*img_gray.shape, 3), dtype=np.uint8) * 255
    for cnt in contours:
        cv2.drawContours(canvas_1, [cnt], -1, color=(0, 0, 0), thickness=3)

    canvas_2 = cv2.GaussianBlur(cv2.cvtColor(canvas_1, cv2.COLOR_BGR2GRAY), (blurrange, blurrange), sigmaX=10000)
    _, canvas_3 = cv2.threshold(canvas_2, 120, 255, cv2.THRESH_BINARY)
    wallContours, _ = cv2.findContours(canvas_3, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    #contour interpolation
    canvas_4 = np.ones((*img_gray.shape, 3), dtype=np.uint8) * 255
    for cnt in wallContours:
        epsilon = e * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.drawContours(canvas_4, [approx], -1, color=(0, 0, 0), thickness=thickness)

    #postprocessing
    n_labels, labels = cv2.connectedComponents(cv2.cvtColor(canvas_4, cv2.COLOR_BGR2GRAY), connectivity=4)
    palette = np.ones_like(canvas_4) * 255
    h, w = labels.shape
    Y, X = np.indices((h, w))
    slope_blank = 10
    slashpos = random.randrange(0, slope_blank)
    fill_style = random.choice(["lslash", "rslash"]) if fillshape == "random" else fillshape

    for label in range(n_labels):
        region = (labels == label)
        gray_count = np.sum((canvas_3 < 200) & region)
        white_count = np.sum((canvas_3 > 200) & region)
        if gray_count > white_count:
            if fill_style == "lslash":
                slash_mask = region & ((X - Y) % slope_blank == slashpos)
            elif fill_style == "rslash":
                slash_mask = region & ((X + Y) % slope_blank == slashpos)
            else:
                slash_mask = region
            palette[region] = (255, 255, 255)
            palette[slash_mask] = SLASH_COLOR
        else:
            palette[region] = (255, 255, 255)


    mask_line = (cv2.cvtColor(canvas_4, cv2.COLOR_BGR2GRAY) == 0)
    palette[mask_line] = (0, 0, 0)
    canvas_4 = palette

    shape = random.choice(["original", "circle", "square", "star"]) if batshape == "random" else batshape

    for cnt in batContours:
        if cv2.contourArea(cnt) > 10000:
            continue
        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        if shape == "original":
            cv2.drawContours(canvas_4, [cnt], -1, color=BAT_COLOR, thickness=-1)
        elif shape == "circle":
            cv2.circle(canvas_4, (cx, cy), radius=6, color=BAT_COLOR, thickness=-1)
        elif shape == "square":
            size = 8
            top_left = (cx - size // 2, cy - size // 2)
            bottom_right = (cx + size // 2, cy + size // 2)
            cv2.rectangle(canvas_4, top_left, bottom_right, color=BAT_COLOR, thickness=-1)
        elif shape == "star":
            size = 6
            pts = []
            for i in range(10):
                angle_deg = i * 36
                angle_rad = math.radians(angle_deg - 90)
                radius = size if i % 2 == 0 else size * 0.4
                x = int(cx + radius * math.cos(angle_rad))
                y = int(cy + radius * math.sin(angle_rad))
                pts.append((x, y))
            pts = np.array(pts, np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(canvas_4, [pts], color=BAT_COLOR)

    return canvas_4


In [14]:
def dirtysketch(img, thresh=93, thickness=2, polyperlength=0.03, s=5.0, batshape="random", fillshape="random"):
    #preprocessing
    img_small = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LANCZOS4)
    img_gray = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY)
    _, mask_bat = cv2.threshold(img_gray, 60, 255, cv2.THRESH_BINARY)

    grid_size = 16
    cell_height = mask_bat.shape[0] // grid_size
    cell_width = mask_bat.shape[1] // grid_size

    for i in range(grid_size):
        y = i * cell_height
        cv2.line(mask_bat, (0, y), (mask_bat.shape[1], y), color=255, thickness=1)
        x = i * cell_width
        cv2.line(mask_bat, (x, 0), (x, mask_bat.shape[0]), color=255, thickness=1)
    cv2.line(mask_bat, (0, mask_bat.shape[0] - 1), (mask_bat.shape[1], mask_bat.shape[0] - 1), color=255, thickness=1)
    cv2.line(mask_bat, (mask_bat.shape[1] - 1, 0), (mask_bat.shape[1] - 1, mask_bat.shape[0]), color=255, thickness=1)

    batContours, _ = cv2.findContours(mask_bat, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    img_gray = cv2.GaussianBlur(img_gray, (3, 3), sigmaX=30, sigmaY=30)
    _, mask_wall = cv2.threshold(img_gray, thresh, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask_wall, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    canvas_1 = np.ones((*img_gray.shape, 3), dtype=np.uint8) * 255
    for cnt in contours:
        cv2.drawContours(canvas_1, [cnt], -1, color=(0, 0, 0), thickness=5)

    border = cell_width
    top_fill = border // 4
    canvas_1[:top_fill, :] = canvas_1[top_fill:top_fill+1, :]
    canvas_1[:, -top_fill:] = canvas_1[:, -top_fill-1:-top_fill]
    canvas_1_1 = cv2.copyMakeBorder(canvas_1, border, border, border, border, cv2.BORDER_REPLICATE)
    canvas_1_2 = cv2.copyMakeBorder(canvas_1_1, border, border, border, border, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    canvas_2 = cv2.GaussianBlur(canvas_1_2, (5, 5), sigmaX=5)
    _, canvas_3_1 = cv2.threshold(canvas_2, 110, 255, cv2.THRESH_BINARY)
    canvas_3 = cv2.cvtColor(canvas_3_1, cv2.COLOR_BGR2GRAY)
    wallContours, _ = cv2.findContours(canvas_3, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    wallContours = [cnt - [2*border, 2*border] for cnt in wallContours]
    canvas_3= canvas_3[2*border:2*border+224, 2*border:2*border+224]

    #contour interpolation
    canvas_4 = np.ones((*img_gray.shape, 3), dtype=np.uint8) * 255
    for cnt in wallContours:
        cnt = augment_contour_points(cnt, max_seg_length=1*cell_width)
        cnt = cnt.squeeze()
        if cnt.ndim != 2 or cnt.shape[1] != 2:
            continue
        x, y = cnt[:, 0], cnt[:, 1]
        try:
            length = cv2.arcLength(cnt, True)
            num_points = max(int(length * polyperlength), 10)
            if length < 8 * (cell_width + cell_height):
                tck, _ = splprep([x, y], s=s * 0.05, per=True)
                u_fine = np.linspace(0, 1, max(num_points, 10))
            else:
                tck, _ = splprep([x, y], s=s, per=True)
                u_fine = np.linspace(0, 1, num_points)
            x_smooth, y_smooth = splev(u_fine, tck)
            pts = np.vstack((x_smooth, y_smooth)).astype(np.int32).T
            cv2.polylines(canvas_4, [pts], isClosed=True, color=(0, 0, 0), thickness=thickness)
        except:
            continue

    #postprocessing
    n_labels, labels = cv2.connectedComponents(cv2.cvtColor(canvas_4, cv2.COLOR_BGR2GRAY), connectivity=4)
    palette = np.ones_like(canvas_4) * 255
    h, w = labels.shape
    Y, X = np.indices((h, w))
    slope_blank = 10
    slashpos = random.randrange(0, slope_blank)
    fill_style = random.choice(["lslash", "rslash"]) if fillshape == "random" else fillshape

    for label in range(n_labels):
        region = (labels == label)
        gray_count = np.sum((canvas_3 < 200) & region)
        white_count = np.sum((canvas_3 > 200) & region)
        if gray_count > white_count:
            if fill_style == "lslash":
                slash_mask = region & ((X - Y) % slope_blank == slashpos)
            elif fill_style == "rslash":
                slash_mask = region & ((X + Y) % slope_blank == slashpos)
            else:
                slash_mask = region
            palette[region] = (255, 255, 255)
            palette[slash_mask] = SLASH_COLOR
        else:
            palette[region] = (255, 255, 255)

    mask_line = (cv2.cvtColor(canvas_4, cv2.COLOR_BGR2GRAY) == 0)
    palette[mask_line] = (0, 0, 0)
    canvas_4 = palette

    shape = random.choice(["original", "circle", "square", "star"]) if batshape == "random" else batshape

    for cnt in batContours:
        if cv2.contourArea(cnt) > 10000:
            continue
        M = cv2.moments(cnt)
        if M["m00"] == 0: continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        if shape == "original":
            cv2.drawContours(canvas_4, [cnt], -1, color=BAT_COLOR, thickness=-1)
        elif shape == "circle":
            cv2.circle(canvas_4, (cx, cy), radius=6, color=BAT_COLOR, thickness=-1)
        elif shape == "square":
            size = 8
            top_left = (cx - size // 2, cy - size // 2)
            bottom_right = (cx + size // 2, cy + size // 2)
            cv2.rectangle(canvas_4, top_left, bottom_right, color=BAT_COLOR, thickness=-1)
        elif shape == "star":
            size = 6
            pts = []
            for i in range(10):
                angle = math.radians(i * 36 - 90)
                radius = size if i % 2 == 0 else size * 0.4
                x = int(cx + radius * math.cos(angle))
                y = int(cy + radius * math.sin(angle))
                pts.append((x, y))
            pts = np.array(pts, np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(canvas_4, [pts], color=BAT_COLOR)

    return canvas_4

In [15]:
def transfer(path, sketchstyle="random", batstyle="random", wallstyle="random"):
    img = cv2.imread(path)

    if sketchstyle == "random":
        style=random.choice([
            "clear_angular",
            "clear_round",
            "dirty_angular",
            "dirty_round"
        ])

    if sketchstyle == "clear_angular":
        img=clearsketch(img, thresh=120, blurrange=7, e=0.001, batshape=batstyle, fillshape=wallstyle,thickness=2)
    elif sketchstyle == "clear_round":
        img=clearsketch(img, thresh=120, blurrange=11, e=0.001, batshape=batstyle, fillshape=wallstyle, thickness=2)
    elif sketchstyle == "dirty_angular":
        img=dirtysketch(img, thresh=110, polyperlength=0.1, s=200.0, batshape=batstyle, fillshape=wallstyle, thickness=2)
    elif sketchstyle == "dirty_round":
        img=dirtysketch(img, thresh=110, polyperlength=1.0, s=400.0, batshape=batstyle, fillshape=wallstyle, thickness=2)
    else:
        print("Unknown sketchstyle")


    return img



In [16]:
from tqdm import tqdm
import os

TRANS_IN_FOLDER=True

base_path=LOAD_FOLDER
save_path_top=SAVE_FOLDER

for top_dir in tqdm(os.listdir(base_path), desc=base_path):
    top_path = os.path.join(base_path, top_dir)
    save_path = os.path.join(save_path_top, top_dir)

    if not os.path.isdir(top_path):
        continue
    elif TRANS_IN_FOLDER==True:
        for file_name in os.listdir(top_path):
            file_path = os.path.join(top_path, file_name)
            save_path= os.path.join(save_path_top, top_dir, file_name)
            if not file_path.lower().endswith(".png"):
                continue
            img=transfer(file_path, sketchstyle="random", batstyle="random")
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            cv2.imwrite(save_path, img)

png: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]
